# Lesson 4: Temperature

Controls how predictable vs creative the reply is. Claude uses 0–1. Gemini uses `config.temperature` (0–2; we'll use 0.0 and 1.0 like the course).

- **0.0–0.3** — facts, code, extraction
- **0.4–0.7** — summaries, teaching
- **0.8–1.0** — brainstorm, jokes, marketing

In [ ]:
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from google.genai.errors import ServerError

load_dotenv(override=True)

client = genai.Client()
model = "gemini-3.6-flash"

`chat` now takes `temperature` (default 1.0).

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "parts": [{"text": text}]})


def generate_with_retry(**params):
    for attempt in range(5):
        try:
            return client.models.generate_content(**params)
        except ServerError:
            if attempt == 4:
                raise
            time.sleep(2 ** attempt)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "contents": messages,
    }

    config_kwargs = {
        "max_output_tokens": 1000,
        "temperature": temperature,
    }
    if system:
        config_kwargs["system_instruction"] = system

    params["config"] = types.GenerateContentConfig(**config_kwargs)
    return generate_with_retry(**params).text

Same prompt twice at `0.0` (should look similar) then twice at `1.0` (more variety).

In [ ]:
prompt = "Give me one original movie idea in one sentence."

print("temperature=0.0")
for i in range(2):
    messages = []
    add_user_message(messages, prompt)
    print(f"{i + 1}.", chat(messages, temperature=0.0))

print("\ntemperature=1.0")
for i in range(2):
    messages = []
    add_user_message(messages, prompt)
    print(f"{i + 1}.", chat(messages, temperature=1.0))